In [3]:
!python3 ../src/featurise.py

[featurise] scikit-learn version: 1.0.1
[featurise] Loading preprocessed CSV from: /home/jovyan/my-work/mlhttp/notebooks/../data/processed/csic_2010_processed.csv
[featurise] Validating input...
[featurise] Creating binary target...
[featurise] Adding derived features...
[featurise] Dropping constant columns...
[featurise] One-hot encoding categorical columns...
[featurise] Selecting model feature columns...
[featurise] Finalizing feature matrix...
[featurise] Input shape: (61065, 57)
[featurise] Feature matrix shape: (61065, 48)
[featurise] Target shape: (61065, 2)
[featurise] Positive class count (anom=1): 25065
[featurise] Negative class count (norm=0): 36000
[featurise] Dropped constant columns: ['protocol', 'user_agent', 'pragma', 'cache_control', 'accept', 'accept_encoding', 'accept_charset', 'accept_language', 'connection']
[featurise] Feature preview:
   content_length  ...  content_type_application/x-www-form-urlencoded
0             0.0  ...                                   

Input: 61,065 requests × 57 columns (after preprocessing)  
Output features: 61,065 × 48 numeric columns  
Target: 61,065 rows with y (25,065 anomalous, 36,000 normal)  
Dropped constant columns: protocol, user_agent, pragma, cache_control, accept, accept_encoding, accept_charset, accept_language, connection  
One-hot encoded: method and content_type (hence columns like content_type_application/x-www-form-urlencoded)  

In [4]:
 # Load outputs
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path().resolve()  # actual notebook path
TASK_BASE = NOTEBOOK_DIR.parent  # relative to notebook path

FEATURES_PATH = TASK_BASE / "data" / "features" / "csic_2010_features.csv"
TARGET_PATH = TASK_BASE / "data" / "features" / "csic_2010_target.csv"
METADATA_PATH = TASK_BASE / "data" / "features" / "csic_2010_feature_metadata.csv"

print("Features:", FEATURES_PATH)
print("Target:", TARGET_PATH)
print("Metadata:", METADATA_PATH)

X = pd.read_csv(FEATURES_PATH)
y = pd.read_csv(TARGET_PATH)
meta = pd.read_csv(METADATA_PATH)

print("\nShapes:")
print("  X:", X.shape)
print("  y:", y.shape)
print("  meta:", meta.shape)

Features: /home/jovyan/my-work/mlhttp/data/features/csic_2010_features.csv
Target: /home/jovyan/my-work/mlhttp/data/features/csic_2010_target.csv
Metadata: /home/jovyan/my-work/mlhttp/data/features/csic_2010_feature_metadata.csv

Shapes:
  X: (61065, 48)
  y: (61065, 2)
  meta: (114, 3)


In [5]:
# Basic sanity checks
print("Feature dtypes:\n", X.dtypes.value_counts())
print("\nAny NaNs in X?", X.isna().any().any())
print("Any NaNs in y?", y.isna().any().any())

print("\nTarget value counts:\n", y["y"].value_counts().sort_index())

print("\nMetadata status counts:\n", meta["status"].value_counts())

Feature dtypes:
 int64      42
float64     6
dtype: int64

Any NaNs in X? False
Any NaNs in y? False

Target value counts:
 0    36000
1    25065
Name: y, dtype: int64

Metadata status counts:
 input               57
output_feature      43
dropped_constant     9
encoded_feature      5
Name: status, dtype: int64


In [6]:
# Feature audit
import numpy as np

n_rows, n_cols = X.shape
zero_var_cols = [col for col in X.columns if X[col].nunique() == 1]
mostly_zero_cols = [
    col for col in X.columns
    if (X[col] == 0).mean() > 0.95
]

print(f"Total features: {n_cols}")
print(f"Constant (zero-variance) features: {len(zero_var_cols)}")
print(f"Mostly-zero (>95% zeros) features: {len(mostly_zero_cols)}")

if zero_var_cols:
    print("\nConstant columns:")
    print(zero_var_cols)

if mostly_zero_cols:
    print("\nMostly-zero columns:")
    print(mostly_zero_cols)

Total features: 48
Constant (zero-variance) features: 12
Mostly-zero (>95% zeros) features: 21

Constant columns:
['has_content_length', 'has_content_type', 'cookie_length', 'url_question_count', 'url_ampersand_count', 'url_equals_count', 'url_percent_count', 'url_parameter_count', 'cookie_semicolon_count', 'cookie_equals_count', 'url_has_query', 'has_path_traversal']

Mostly-zero columns:
['url_question_count', 'url_ampersand_count', 'url_equals_count', 'url_percent_count', 'url_parameter_count', 'payload_angle_bracket_count', 'payload_paren_count', 'payload_semicolon_count', 'payload_comment_count', 'cookie_semicolon_count', 'url_has_query', 'has_sql_select', 'has_sql_union', 'has_sql_drop', 'has_sql_delete', 'has_script_tag', 'has_alert', 'has_or_true', 'has_comment', 'has_path_traversal', 'method_PUT']


In [8]:
# Spot-check a few rows
print(X.head())

   content_length  has_content_length  has_content_type  has_payload  \
0             0.0                   1                 1            1   
1             0.0                   1                 1            1   
2             0.0                   1                 1            0   
3             0.0                   1                 1            1   
4             0.0                   1                 1            1   

   url_length  payload_length  cookie_length  content_type_length  \
0          48             146             43                    0   
1          48              77             43                    0   
2          40               0             43                    0   
3          52             118             43                    0   
4          52              69             43                    0   

   url_slash_count  url_dot_count  ...  has_script_tag  has_alert  \
0                5              1  ...               0          0   
1             

In [9]:
print(y.head())

  request_id  y
0     anom_0  1
1     anom_1  1
2     anom_2  1
3     anom_3  1
4     anom_4  1


In [10]:
# Summary:
assert X.shape[0] == y.shape[0], "Row count mismatch between X and y"
assert not X.isna().any().any(), "NaNs found in feature matrix"
assert set(y["y"].unique()) <= {0, 1}, "Target must be binary 0/1"
assert all(X.dtypes.apply(lambda d: np.issubdtype(d, np.number))), "Non-numeric features remain"

print("All assertions passed — feature matrix is model-ready.")

All assertions passed — feature matrix is model-ready.
